<h1 style="text-align: center; margin: 0; padding: 20px; background-color: #2a3128; color: white;">Home Credit - Robust Feature Engineering</h1>

#### The primary goal of this notebook is to predict the probability of loan default for **Home Credit**, specifically focusing on "Unbanked" populations.

<div style="width: 100%;">
    <img style="width: 100%;" src="https://i.ytimg.com/vi/UomcrYhcefw/hq720.jpg?sqp=-oaymwE7CK4FEIIDSFryq4qpAy0IARUAAAAAGAElAADIQj0AgKJD8AEB-AH-DoACuAiKAgwIABABGDwgWShlMA8=&rs=AOn4CLCgLbnq8-z-A-Gs-GhKWFJbLHG0Gw")/>
</div>

<h1 style="margin: 0; padding: 20px; background-color: #2a3128; color: white; text-align: left;">1. Import necessary libraries</h1>

In [ ]:
import numpy as np
import pandas as pd
import gc, re, warnings, os
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
from scipy.stats import rankdata
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize

<h1 style="margin: 0; padding: 20px; background-color: #2a3128; color: white; text-align: left;">2. Environment Setup</h1>

In [ ]:
# ignore warnings
warnings.filterwarnings('ignore')

In [ ]:
# max number of columns pandas can show
pd.set_option('display.max_columns', 200)

In [ ]:
# get data 
DATA_DIR = Path("/kaggle/input/home-credit-default-risk")
if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input/competitions/home-credit-default-risk")

In [ ]:
# Number of folds for Stratified Cross-Validation
N_FOLDS = 5

In [ ]:
# Deterministic seed for reproducibility across different runs
SEED = 42

#### Target Encoding Parameters:

In [ ]:
# TE_SMOOTHING: Controls the weight given to the category average vs. the global average.
# Higher values reduce the risk of overfitting in small categories --> go to global average.
TE_SMOOTHING = 40 

In [ ]:
# TE_MIN_SAMPLES: The minimum number of observations required in a category 
# to rely on its own mean instead of the global average 
# and if it's go to global avg the model will dep on Frequency Encoding
TE_MIN_SAMPLES = 80

In [ ]:
# check if i have gpu or not
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False
print(f"GPU available: {HAS_GPU}")

In [ ]:
# Reads a CSV file and optimizes memory usage by downcasting data types while reading to avoid crash
def read_csv(path, usecols=None):
    df = pd.read_csv(path, usecols=usecols)
    for col in df.columns:
        if df[col].dtype == "float64":
            df[col] = df[col].astype("float32")
        elif df[col].dtype == "int64":
            # avoid Integer Overflow
            if df[col].min() >= np.iinfo(np.int32).min and df[col].max() <= np.iinfo(np.int32).max:
                df[col] = df[col].astype("int32")
    return df
    
#  downcasts numeric types to the smallest possible representation after feature engineering
def reduce_memory_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if str(col_type).startswith('float'):
            df[col] = pd.to_numeric(df[col], downcast='float')
        elif str(col_type).startswith('int'):
            df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

<h1 style="margin: 0; padding: 20px; background-color: #2a3128; color: white; text-align: left;">3. Helper Functions</h1>

$$w = e^{\lambda \cdot t}$$

Where:
* **$w$**: The resulting weight (importance) of the record.
* **$\lambda$ (Decay)**: The decay constant 
* **$t$ (Time)**: The time variable 0 mean today -1 yesterday etc..

**If Decay** == 0.002 :
+ Today ($t = 0$): $e^0 = 1$. The weight is 100%. We trust this data completely.
+ Past ($t = -1000$): $e^{-2}$ is about 0.13. The weight is only 13%. We barely trust this data.

**A greater decay value means a greater focus on the present**

In [ ]:
# Calculates Time-Weighted Average for features using Exponential Decay.
# Recent events get higher weights than older events.

def time_weighted_agg(df, group_col, value_cols, time_col, prefix, decay=0.002):
    
    ids = df[group_col].unique()
    result = pd.DataFrame({group_col: ids})
   
    w = np.exp(decay * df[time_col].values.astype('float64'))

    # loop calculate weighted aggregation and ignore null
    for vc in value_cols:
        
        mask = df[vc].notna().values
        if mask.sum() == 0:
            result[f'{prefix}_{vc}_TWMEAN'] = np.nan
            continue
            
        grp_vals = df[group_col].values[mask]
        wv = df[vc].values[mask].astype('float64') * w[mask]
        ww = w[mask]
        
        temp = pd.DataFrame({group_col: grp_vals, '_wv': wv, '_w': ww})
        agg = temp.groupby(group_col)[['_wv', '_w']].sum()
        agg[f'{prefix}_{vc}_TWMEAN'] = (agg['_wv'] / agg['_w']).astype('float32')
        
        result = result.merge(agg[[f'{prefix}_{vc}_TWMEAN']].reset_index(), on=group_col, how='left')
    
    return result

$$\beta = \frac{\sum (t - \bar{t})(v - \bar{v})}{\sum (t - \bar{t})^2}$$

Where:
* **$t$**: Time (Days)
* **$v$**: Value (e.g., Loan Amount, Installment)
* **$\bar{t}, \bar{v}$**: The mean of time and values for that specific customer.

**cov(v,t)/ var(t)**

In [ ]:
def compute_trend(df, group_col, value_col, time_col, prefix):
    # Select only needed columns and drop rows with missing values
    temp = df[[group_col, value_col, time_col]].dropna().copy()

    # If no data left, return NaN trend for all groups
    if len(temp) == 0:
        return pd.DataFrame({
            group_col: df[group_col].unique(),
            f'{prefix}_{value_col}_TREND': np.nan
        })

    # Count number of records per group
    gcounts = temp.groupby(group_col)[value_col].transform('count')

    # Keep only groups with at least 3 observations (needed for trend)
    temp = temp[gcounts >= 3].copy()

    # If no valid groups remain, return NaN trend
    if len(temp) == 0:
        return pd.DataFrame({
            group_col: df[group_col].unique(),
            f'{prefix}_{value_col}_TREND': np.nan
        })

    # Group data by group_col
    g = temp.groupby(group_col)

    # Compute mean time and mean value per group
    mt = g[time_col].transform('mean').astype('float64')   # mean time
    mv = g[value_col].transform('mean').astype('float64')  # mean value

    # Compute deviations from mean
    dt = temp[time_col].astype('float64') - mt  # time deviation
    dv = temp[value_col].astype('float64') - mv # value deviation

    # Compute components for slope (covariance / variance)
    temp['_dtdv'] = dt * dv   # covariance numerator
    temp['_dt2'] = dt ** 2    # variance denominator

    # Aggregate sums per group
    agg = temp.groupby(group_col)[['_dtdv', '_dt2']].sum()

    # Compute trend (slope of value over time)
    agg[f'{prefix}_{value_col}_TREND'] = (
        agg['_dtdv'] / (agg['_dt2'] + 1e-8)  # avoid division by zero
    ).astype('float32')

    # Return only group and computed trend
    return agg[[f'{prefix}_{value_col}_TREND']].reset_index()


In [ ]:
# Creates relative features by comparing individual values to group statistics.
    
def add_groupby_ratio_features(train_df, test_df, cat_cols, num_cols):
    
    full = pd.concat([train_df.drop(columns=['TARGET'], errors='ignore'), test_df], axis=0, ignore_index=True)
    
    for cat in cat_cols:
        if cat not in full.columns:
            continue
        for num in num_cols:
            if num not in full.columns:
                continue
            # Compute Group Statistics 
            gp = full.groupby(cat)[num].agg(['mean', 'median', 'std']).reset_index()
           
            gp.columns = [cat,
                          f'GB_{cat}_{num}_MEAN',
                          f'GB_{cat}_{num}_MEDIAN',
                          f'GB_{cat}_{num}_STD']
            
            full = full.merge(gp, on=cat, how='left')
            
            # Calculate the distance from the group average
            full[f'GB_{cat}_{num}_DIFF'] = full[num] - full[f'GB_{cat}_{num}_MEAN']
            # Relative ratio to mean 
            full[f'GB_{cat}_{num}_RATIO'] = full[num] / (full[f'GB_{cat}_{num}_MEAN'] + 1e-6)
            
    full = reduce_memory_usage(full)
    
    out_train = full.iloc[:len(train_df)].copy()
    out_test = full.iloc[len(train_df):].copy()
    
    if 'TARGET' in train_df.columns:
        out_train['TARGET'] = train_df['TARGET'].values
    
    return out_train, out_test


In [ ]:
# Focuses on a specific recent time window (e.g., last 120 days).
# id,mean,sum,max,count

def agg_time_window(df, group_col, cols, time_col, cutoff, prefix):
    
    sub = df[df[time_col] >= cutoff]
    
    if len(sub) == 0:
        return pd.DataFrame({group_col: df[group_col].unique()})
    
    valid_cols = [c for c in cols if c in sub.columns]
    if not valid_cols:
        return pd.DataFrame({group_col: df[group_col].unique()})
        
    agg = sub.groupby(group_col)[valid_cols].agg(['mean', 'max', 'sum'])
    
    # Flatten the multi-index columns (e.g., 'AMT_CREDIT', 'mean' -> 'prefix_AMT_CREDIT_MEAN')
    agg.columns = [f'{prefix}_{c[0]}_{c[1].upper()}' for c in agg.columns]
   
    cnt = sub.groupby(group_col).size().reset_index(name=f'{prefix}_COUNT')
    
    agg = agg.reset_index().merge(cnt, on=group_col, how='left')
    
    return agg

In [ ]:
# add frequency of occurrence & that is the way we handle null in categorical

def add_frequency_features(train_df, test_df, cols):
    
    full = pd.concat([train_df.drop(columns=['TARGET'], errors='ignore'), test_df], axis=0, ignore_index=True)
   
    for col in cols:
        if col not in full.columns:
            continue
            
        # ensures that missing values are also counted as a distinct group 
        vc = full[col].fillna('__nan__').value_counts(dropna=False)
        
        full[f'{col}_FREQ'] = full[col].fillna('__nan__').map(vc).astype('float32')
        full[f'{col}_FREQ_NORM'] = (full[f'{col}_FREQ'] / len(full)).astype('float32')
        
    out_train = full.iloc[:len(train_df)].copy()
    out_test = full.iloc[len(train_df):].copy()
    
    if 'TARGET' in train_df.columns:
        out_train['TARGET'] = train_df['TARGET'].values
        
    return out_train, out_test

In [ ]:
# Target Encoding
def add_target_encoding(train_df, test_df, target_col, cols, n_splits=N_FOLDS, smoothing=TE_SMOOTHING, min_samples_leaf=TE_MIN_SAMPLES, seed=SEED):
    
    train_df = train_df.copy()
    test_df = test_df.copy()
   
    global_mean = train_df[target_col].mean()
    
    # ensure each fold has the same ratio of Target 0 and 1 because target is imbalanced
    # and i want to avoid overfit so i will make fold because i want to give category diff value
    # over diff folds to make model git the range that category move between it and not specific number
    # and in test we will give each category generanl number which is trained over all data
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    for col in cols:
        
        if col not in train_df.columns:
            continue
            
        new_col = f'{col}_TE'
        
        if new_col in train_df.columns:
            continue  # Skip if already exists

        # Initialize arrays to store encoded values
        tr_enc = np.zeros(len(train_df), dtype='float32')
        te_enc = np.zeros(len(test_df), dtype='float32')
        
        # Set Encoding (Out-of-Fold logic)
        for tr_idx, va_idx in skf.split(train_df, train_df[target_col]):
            
            tr_fold = train_df.iloc[tr_idx]
            
            stats = tr_fold.groupby(col)[target_col].agg(['mean', 'count'])
            
            # Weighted average between the category mean and the global bank mean
            smooth = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
            # If a category has too few samples, use the global mean instead
            smooth[stats['count'] < min_samples_leaf] = global_mean
            # map smooth value to validation fold and fill null with global mean
            tr_enc[va_idx] = train_df.iloc[va_idx][col].map(smooth).fillna(global_mean).values.astype('float32')
        
        # Test Set Encoding on all train data not just fold
        full_stats = train_df.groupby(col)[target_col].agg(['mean', 'count'])
        full_smooth = (full_stats['count'] * full_stats['mean'] + smoothing * global_mean) / (full_stats['count'] + smoothing)
        full_smooth[full_stats['count'] < min_samples_leaf] = global_mean
        
        te_enc = test_df[col].map(full_smooth).fillna(global_mean).values.astype('float32')
        
        train_df[new_col] = tr_enc
        test_df[new_col] = te_enc
    
    return train_df, test_df

<h1 style="margin: 0; padding: 20px; background-color: #2a3128; color: white; text-align: left;">4. Feature Engineering</h1>

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">1. Application Table Features</h2>

In [ ]:
def application_features(df):
    
    out = df.copy()
    
    # replace outlier value with null and create new col to boolean flag
    out['DAYS_EMPLOYED'] = out['DAYS_EMPLOYED'].replace(365243, np.nan)
    out['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype('int8')

    # we will count columns for each row 
    def _row_sum_numeric(frame, cols, dtype='float32'):
        cols = [c for c in cols if c in frame.columns]
        if not cols:
            return pd.Series(np.zeros(len(frame), dtype=dtype), index=frame.index)
        block = frame[cols].apply(pd.to_numeric, errors='coerce')
        return block.sum(axis=1).astype(dtype)

   
    # Select external source features
    
    ext = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    
    # gplearn (genatic programing) used here  and some based on domain understanding
    
    out['EXT_MEAN'] = out[ext].mean(axis=1)  # Average risk score
    out['EXT_STD']  = out[ext].std(axis=1)   # Variation between scores (consistency)
    out['EXT_PROD'] = out['EXT_SOURCE_1'] * out['EXT_SOURCE_2'] * out['EXT_SOURCE_3'] # Combined multiplicative effect
    out['EXT_MIN']  = out[ext].min(axis=1)   # Worst score (most risky)
    out['EXT_MAX']  = out[ext].max(axis=1)   # Best score (least risky)
    out['EXT_NANCOUNT'] = out[ext].isna().sum(axis=1)  # Number of missing values
    
    # Pairwise interactions between external scores
    out['EXT_S1xS2'] = out['EXT_SOURCE_1'] * out['EXT_SOURCE_2'] # Interaction between S1 and S2
    out['EXT_S1xS3'] = out['EXT_SOURCE_1'] * out['EXT_SOURCE_3'] # Interaction between S1 and S3
    out['EXT_S2xS3'] = out['EXT_SOURCE_2'] * out['EXT_SOURCE_3'] # Interaction between S2 and S3
    
    # Ratios between scores
    out['EXT_S2divS3'] = out['EXT_SOURCE_2'] / (out['EXT_SOURCE_3'] + 1e-4)
    out['EXT_S1divS2'] = out['EXT_SOURCE_1'] / (out['EXT_SOURCE_2'] + 1e-4)

    # capture non-linear relationships
    for i in [1, 2, 3]:
        col = f'EXT_SOURCE_{i}'
        out[f'{col}_SQ'] = out[col] ** 2
        out[f'{col}_CB'] = out[col] ** 3

    # Interaction with age Score influenced by age
    out['EXT_S2xBIRTH'] = out['EXT_SOURCE_2'] * out['DAYS_BIRTH']
    out['EXT_S1xBIRTH'] = out['EXT_SOURCE_1'] * out['DAYS_BIRTH']
    out['EXT_S3xBIRTH'] = out['EXT_SOURCE_3'] * out['DAYS_BIRTH']

    # Interaction with employment duration Score influenced by job stability
    out['EXT_S2xEMPL']  = out['EXT_SOURCE_2'] * out['DAYS_EMPLOYED']
    out['EXT_S3xEMPL']  = out['EXT_SOURCE_3'] * out['DAYS_EMPLOYED']
    
    # gplearn
    out['GP1'] = out['EXT_SOURCE_2'] ** 2 * out['EXT_SOURCE_3']
    out['GP2'] = out['EXT_SOURCE_1'] * out['DAYS_BIRTH'] / (out['AMT_ANNUITY'] + 1)
    out['GP3'] = out['EXT_SOURCE_2'] * out['REGION_RATING_CLIENT_W_CITY']
    out['GP4'] = out['EXT_SOURCE_3'] * np.log1p(np.abs(out['DAYS_BIRTH']))
    out['GP5'] = out['AMT_ANNUITY'] * out['EXT_SOURCE_3'] / (out['AMT_INCOME_TOTAL'] + 1)
    out['GP6'] = out['EXT_SOURCE_1'] * out['DAYS_ID_PUBLISH'] / (out['DAYS_BIRTH'] + 1)
    out['GP7'] = out['EXT_SOURCE_2'] * out['AMT_CREDIT'] / (out['AMT_GOODS_PRICE'] + 1)
    out['GP8'] = out['EXT_SOURCE_1'] * out['EXT_SOURCE_2'] * out['EXT_SOURCE_3'] / (out['AMT_CREDIT'] + 1)
    out['GP9'] = out['EXT_MEAN'] * out['DAYS_EMPLOYED'] / (out['DAYS_BIRTH'] - 1)
    out['GP10'] = (out['AMT_GOODS_PRICE'] - out['AMT_CREDIT']) * out['EXT_SOURCE_2'] / (out['AMT_ANNUITY'] + 1)

    # Financial ratios domain knowledge
    # How large the credit is relative to income (higher = riskier)
    out['CREDIT_INCOME_RATIO']  = out['AMT_CREDIT'] / (out['AMT_INCOME_TOTAL'] + 1)
    # Monthly payment burden relative to income (affordability)
    out['ANNUITY_INCOME_RATIO'] = out['AMT_ANNUITY'] / (out['AMT_INCOME_TOTAL'] + 1)
    # Approximate loan duration (total credit divided by periodic payment)
    out['CREDIT_ANNUITY_RATIO'] = out['AMT_CREDIT'] / (out['AMT_ANNUITY'] + 1)
    # Portion of goods price covered by credit (loan coverage level)
    out['CREDIT_GOODS_RATIO']   = out['AMT_CREDIT'] / (out['AMT_GOODS_PRICE'] + 1)
    # Cost of goods relative to income (affordability of purchase)
    out['GOODS_INCOME_RATIO']   = out['AMT_GOODS_PRICE'] / (out['AMT_INCOME_TOTAL'] + 1)
    # Income distributed per child (financial pressure indicator)
    out['INCOME_PER_CHILD']     = out['AMT_INCOME_TOTAL'] / (out['CNT_CHILDREN'] + 1)
    # Income per family member (overall household financial capacity)
    out['INCOME_PER_FAM']       = out['AMT_INCOME_TOTAL'] / (out['CNT_FAM_MEMBERS'] + 1)
    # Payment relative to total credit (inverse of loan length intuition)
    out['ANNUITY_CREDIT_RATIO'] = out['AMT_ANNUITY'] / (out['AMT_CREDIT'] + 1)
    # Estimated number of payment periods (loan duration proxy)
    out['PAYMENT_LENGTH']       = out['AMT_CREDIT'] / (out['AMT_ANNUITY'] + 1)
    # Initial payment made by the client (own contribution)
    out['DOWN_PAYMENT']         = out['AMT_GOODS_PRICE'] - out['AMT_CREDIT']
    # Percentage of goods price paid upfront (higher = lower risk)
    out['DOWN_PAYMENT_RATIO']   = out['DOWN_PAYMENT'] / (out['AMT_GOODS_PRICE'] + 1)

    # Age & Employment
    # Convert age from days to years (original values are negative)
    out['DAYS_BIRTH_YRS']    = out['DAYS_BIRTH'] / -365.25
    # Convert employment duration from days to years
    out['DAYS_EMPLOYED_YRS'] = out['DAYS_EMPLOYED'] / -365.25
    # Ratio of employment duration to age (how much of life spent working)
    out['EMPLOYED_TO_BIRTH'] = out['DAYS_EMPLOYED'] / (out['DAYS_BIRTH'] + 1)
    # Car age relative to person’s age (lifestyle / financial indicator)
    out['CAR_AGE_TO_BIRTH']  = out['OWN_CAR_AGE'] / (out['DAYS_BIRTH_YRS'] + 1)
    # Recency of ID update relative to age (may reflect recent changes)
    out['ID_PUBLISH_TO_BIRTH'] = out['DAYS_ID_PUBLISH'] / (out['DAYS_BIRTH'] + 1)
    # Last phone change relative to age (customer activity / stability)
    out['PHONE_TO_BIRTH']    = out['DAYS_LAST_PHONE_CHANGE'] / (out['DAYS_BIRTH'] + 1)
    # Phone change relative to employment duration (job stability signal)
    out['PHONE_TO_EMPLOYED'] = out['DAYS_LAST_PHONE_CHANGE'] / (out['DAYS_EMPLOYED'] + 1)
    # Registration duration relative to age (customer history length)
    out['REG_TO_BIRTH']      = out['DAYS_REGISTRATION'] / (out['DAYS_BIRTH'] + 1)
    # Bin ages into groups to capture non-linear age effects
    out['AGE_RANGE'] = pd.cut(out['DAYS_BIRTH_YRS'], bins=[0,25,30,35,40,45,50,55,60,65,100], labels=False)
    # Combine income with employment duration (proxy for financial stability)
    out['INCOME_EMPLOYED'] = out['AMT_INCOME_TOTAL'] * out['DAYS_EMPLOYED_YRS']

    # count number of document user provide it
    doc_cols = [c for c in out.columns if 'FLAG_DOCUMENT' in c]
    out['DOCUMENT_COUNT'] = _row_sum_numeric(out, doc_cols, dtype='float32')

    # Ratio of people with payment difficulties within 30 days (risk indicator)
    out['DEF_30_RATIO'] = out['DEF_30_CNT_SOCIAL_CIRCLE'] / (out['OBS_30_CNT_SOCIAL_CIRCLE'] + 1)
    # Ratio of people with payment difficulties within 60 days (longer-term risk indicator)
    out['DEF_60_RATIO'] = out['DEF_60_CNT_SOCIAL_CIRCLE'] / (out['OBS_60_CNT_SOCIAL_CIRCLE'] + 1)
    
    # Total number of missing values per applicant (data quality)
    out['APP_NULLS'] = out.isna().sum(axis=1).astype('int16')
    # Combines regional risk level with external credit score
    out['CITY_RATING_x_EXT2'] = out['REGION_RATING_CLIENT_W_CITY'] * out['EXT_SOURCE_2']

    # Employment duration compared to ID update timing (stability vs identity changes)
    out['EMPLOYED_TO_ID'] = out['DAYS_EMPLOYED'] / (out['DAYS_ID_PUBLISH'] + 1)
    # How recent ID updates are relative to age
    out['ID_TO_BIRTH_RATIO'] = out['DAYS_ID_PUBLISH'] / (out['DAYS_BIRTH'] + 1)
    # Registration history compared to employment duration (profile stability)
    out['REG_TO_EMPLOYED_RATIO'] = out['DAYS_REGISTRATION'] / (out['DAYS_EMPLOYED'] + 1)
    # Credit burden per family member
    out['CREDIT_PER_PERSON'] = out['AMT_CREDIT'] / (out['CNT_FAM_MEMBERS'] + 1)
    # Payment burden per family member
    out['ANNUITY_PER_PERSON'] = out['AMT_ANNUITY'] / (out['CNT_FAM_MEMBERS'] + 1)
    # Income relative to total credit (ability to repay)
    out['INCOME_CREDIT_PERC'] = out['AMT_INCOME_TOTAL'] / (out['AMT_CREDIT'] + 1)
    # Income relative to monthly payment (affordability indicator)
    out['INCOME_ANNUITY_PERC'] = out['AMT_INCOME_TOTAL'] / (out['AMT_ANNUITY'] + 1)
    # Spread between best and worst external scores
    out['EXT_RANGE'] = out['EXT_MAX'] - out['EXT_MIN']
    # Relative variability of external scores (consistency measure)
    out['EXT_SOURCE_SPREAD'] = out['EXT_STD'] / (out['EXT_MEAN'] + 1e-4)
    # Time difference between phone update and registration (activity pattern)
    out['PHONE_MINUS_REG'] = out['DAYS_LAST_PHONE_CHANGE'] - out['DAYS_REGISTRATION']
    # Car age compared to employment duration (lifestyle stability)
    out['CAR_EMPLOYED_RATIO'] = out['OWN_CAR_AGE'] / (out['DAYS_EMPLOYED_YRS'] + 1)
    # Proportion of children in household (dependency level)
    out['CHILDREN_RATIO'] = out['CNT_CHILDREN'] / (out['CNT_FAM_MEMBERS'] + 1)
    # Comparison of observed social defaults in short vs longer window
    out['OBS_30_60_RATIO'] = out['OBS_30_CNT_SOCIAL_CIRCLE'] / (out['OBS_60_CNT_SOCIAL_CIRCLE'] + 1)
    # Comparison of default rates in social circle across time window
    out['DEF_30_60_RATIO'] = out['DEF_30_CNT_SOCIAL_CIRCLE'] / (out['DEF_60_CNT_SOCIAL_CIRCLE'] + 1)
    
    # Total number of credit bureau requests per applicant across all time windows (overall credit-seeking activity indicator
    amt_req_cols = [c for c in out.columns if c.startswith('AMT_REQ_CREDIT_BUREAU_')]
    out['AMT_REQ_SUM'] = _row_sum_numeric(out, amt_req_cols, dtype='float32')

    # Total number of available contact methods for the applicant
    explicit_contact_cols = [
        c for c in ['FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL']
        if c in out.columns
    ]
    if explicit_contact_cols:
        out['FLAG_CONTACTS_SUM'] = _row_sum_numeric(out, explicit_contact_cols, dtype='float32')

    # Proportion of credit paid per installment (proxy for repayment intensity / loan structure)
    out['CREDIT_TERM'] = out['AMT_ANNUITY'] / (out['AMT_CREDIT'] + 1)
    # Fraction of life spent working (employment stability indicator)
    out['DAYS_EMPLOYED_PERC'] = out['DAYS_EMPLOYED'] / (out['DAYS_BIRTH'] + 1)
    # Income relative to credit amount (repayment capacity signal)
    out['INCOME_CREDIT_PERC2'] = out['AMT_INCOME_TOTAL'] / (out['AMT_CREDIT'] + 1)
    # Custom weighted combination of external scores (giving more importance to EXT_SOURCE_2)
    out['EXT_WEIGHTED'] = 2*out['EXT_SOURCE_2'] + out['EXT_SOURCE_3'] + 0.5*out['EXT_SOURCE_1']
    # Interaction between regional population density and external credit score
    out['REGION_POP_x_EXT'] = out['REGION_POPULATION_RELATIVE'] * out['EXT_MEAN']
    # Interaction between application time and external credit score (behavioral pattern signal)
    out['HOUR_APPR_x_EXT2'] = out['HOUR_APPR_PROCESS_START'] * out['EXT_SOURCE_2']
    
    # Measures inconsistencies between registered region, work region, and living region
    # Higher value = more regional mismatch (possible instability or data inconsistency)
    out['LIVE_REGION_DIFF'] = (out['REG_REGION_NOT_LIVE_REGION'].astype(float) +
                                out['REG_REGION_NOT_WORK_REGION'].astype(float) +
                                out.get('LIVE_REGION_NOT_WORK_REGION', pd.Series(0, index=out.index)).astype(float))

    return out

app_train_raw = read_csv(DATA_DIR / "application_train.csv")
app_test_raw  = read_csv(DATA_DIR / "application_test.csv")

app_train = application_features(app_train_raw)
app_test  = application_features(app_test_raw)
# delete the data we don't need it any more and call garbage collector
del app_train_raw, app_test_raw; gc.collect()

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">2. Bureau & Bureau Balance tables features</h2>

In [ ]:

def bureau_and_balance_features():
    
    bureau = read_csv(DATA_DIR / "bureau.csv")
    bb = read_csv(DATA_DIR / "bureau_balance.csv")

    # Bureau Balance Aggregation: Transform monthly history into counts
    
    # pivot_table converts STATUS values (0, 1, C, X, etc.) into separate columns
    bb_counts = bb.pivot_table(index="SK_ID_BUREAU", columns="STATUS",values="MONTHS_BALANCE", aggfunc="count", fill_value=0)
    bb_counts.columns = [f"BB_STATUS_{c}" for c in bb_counts.columns]
    bb_counts = bb_counts.reset_index()
    
    # Calculate time range for each external loan
    bb_months = bb.groupby("SK_ID_BUREAU")["MONTHS_BALANCE"].agg(BB_MONTHS_MIN="min", BB_MONTHS_MAX="max", BB_MONTHS_SIZE="size").reset_index()
    
    bb_agg = bb_months.merge(bb_counts, on="SK_ID_BUREAU", how="left")
    bureau = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")
    # delete the data we don't need it any more and call garbage collector
    del bb, bb_counts, bb_months, bb_agg; gc.collect()

    # Duration: How long the loan was supposed to last
    bureau['CREDIT_DURATION']    = bureau['DAYS_CREDIT_ENDDATE'] - bureau['DAYS_CREDIT']
    # Difference between expected end and actual end date
    bureau['ENDDATE_DIFF']        = bureau['DAYS_CREDIT_ENDDATE'] - bureau['DAYS_ENDDATE_FACT']
    # Debt-to-Credit Ratio (Credit Utilization): Extremely important risk signal
    bureau['DEBT_CREDIT_RATIO']  = bureau['AMT_CREDIT_SUM_DEBT'] / (bureau['AMT_CREDIT_SUM'] + 1)
    # Overdue-to-Debt Ratio: Percentage of current debt that is past due
    bureau['OVERDUE_DEBT_RATIO'] = bureau['AMT_CREDIT_SUM_OVERDUE'] / (bureau['AMT_CREDIT_SUM_DEBT'] + 1)
    # Monthly payment-to-total credit ratio: Measures the 'weight' of the installment relative to the loan size
    bureau['AMT_ANNUITY_CREDIT'] = bureau['AMT_ANNUITY'] / (bureau['AMT_CREDIT_SUM'] + 1)
    # How much of the total credit is overdue
    bureau['CREDIT_OVERDUE_RATIO'] = bureau['AMT_CREDIT_SUM_OVERDUE'] / (bureau['AMT_CREDIT_SUM'] + 1)
    # Difference between last report update and start date
    bureau['DAYS_CREDIT_UPDATE_DIFF'] = bureau['DAYS_CREDIT_UPDATE'] - bureau['DAYS_CREDIT']

    #  Numerical Aggregation: Grouping multiple loans per client
    num_cols = [c for c in bureau.columns if bureau[c].dtype != 'object' and c not in ['SK_ID_BUREAU', 'SK_ID_CURR']]
    #  Calculate stats (min, max, mean, sum, variance) for all numeric bureau features
    buro_num = bureau.groupby('SK_ID_CURR')[num_cols].agg(['min', 'max', 'mean', 'sum', 'var'])
    # rename column
    buro_num.columns = [f'BURO_{c[0]}_{c[1].upper()}' for c in buro_num.columns]
    buro_feat = buro_num.reset_index()

    # Categorical Aggregation: Frequency of loan types (using One-Hot Encoding + Mean)
    cat_cols = [c for c in bureau.columns if bureau[c].dtype == 'object']
    if cat_cols:
        # dummy_na=True ensures that missing values are treated as a separate category
        buro_cat = pd.get_dummies(bureau[['SK_ID_CURR'] + cat_cols], columns=cat_cols, dummy_na=True)
        buro_cat = buro_cat.groupby('SK_ID_CURR').mean().reset_index()
        # rename column
        buro_cat.columns = ['SK_ID_CURR'] + [f'BURO_{c}' for c in buro_cat.columns if c != 'SK_ID_CURR']
        buro_feat = buro_feat.merge(buro_cat, on='SK_ID_CURR', how='left')
    
    # Count of total external loans per client
    buro_feat = buro_feat.merge(bureau.groupby('SK_ID_CURR').size().reset_index(name='BURO_COUNT'),on='SK_ID_CURR', how='left')

    #  Active vs Closed Splits: Isolate current debt from finished debt
    for status in ['Active', 'Closed']:
        sub = bureau[bureau['CREDIT_ACTIVE'] == status]
        if len(sub) > 0:
            key = ['AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'DAYS_CREDIT', 'DAYS_CREDIT_ENDDATE', 'DEBT_CREDIT_RATIO']
            key = [c for c in key if c in sub.columns]
            
            sa = sub.groupby('SK_ID_CURR')[key].agg(['mean', 'sum', 'max', 'min'])
            sa.columns = [f'BURO_{status.upper()}_{c[0]}_{c[1].upper()}' for c in sa.columns]
            sa = sa.reset_index()
            sc = sub.groupby('SK_ID_CURR').size().reset_index(name=f'BURO_{status.upper()}_COUNT')
            
            buro_feat = buro_feat.merge(sa.merge(sc, on='SK_ID_CURR', how='left'),on='SK_ID_CURR', how='left')

    #  Calculate stats for loans taken in last 6M, 1Y, 2Y, etc.
    tw_cols = ['AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'CREDIT_DAY_OVERDUE', 'DEBT_CREDIT_RATIO']
    tw_cols = [c for c in tw_cols if c in bureau.columns]
    for days, label in [(-180, '6M'), (-365, '1Y'), (-730, '2Y'), (-1095, '3Y'), (-1825, '5Y')]:
        tw = agg_time_window(bureau, 'SK_ID_CURR', tw_cols, 'DAYS_CREDIT', days, f'BURO_{label}')
        buro_feat = buro_feat.merge(tw, on='SK_ID_CURR', how='left')

    #  Recent credits have more influence than older ones
    tw_feats = time_weighted_agg(bureau, 'SK_ID_CURR',['AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'DEBT_CREDIT_RATIO'],'DAYS_CREDIT', 'BURO', decay=0.001)
    buro_feat = buro_feat.merge(tw_feats, on='SK_ID_CURR', how='left')

    #  Is the debt growing or shrinking over time?
    for col in ['AMT_CREDIT_SUM_DEBT', 'DEBT_CREDIT_RATIO']: 
        trend = compute_trend(bureau, 'SK_ID_CURR', col, 'DAYS_CREDIT', 'BURO')
        buro_feat = buro_feat.merge(trend, on='SK_ID_CURR', how='left')

    # State of the very last record found in Bureau
    bureau_sorted = bureau.sort_values('DAYS_CREDIT', ascending=False)
    last_bureau = bureau_sorted.groupby('SK_ID_CURR').first().reset_index()
    for col in ['DAYS_CREDIT', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'DEBT_CREDIT_RATIO', 'CREDIT_DAY_OVERDUE']:
        if col in last_bureau.columns:
            buro_feat = buro_feat.merge(
                last_bureau[['SK_ID_CURR', col]].rename(columns={col: f'BURO_LAST_{col}'}),
                on='SK_ID_CURR', how='left')

    # Number of different credit types
    buro_feat = buro_feat.merge(bureau.groupby('SK_ID_CURR')['CREDIT_TYPE'].nunique().reset_index(name='BURO_CREDIT_TYPE_NUNIQUE'),on='SK_ID_CURR', how='left')

    # Flag if the client has EVER had an overdue payment in any external credit
    buro_feat = buro_feat.merge((bureau.groupby('SK_ID_CURR')['CREDIT_DAY_OVERDUE'].max() > 0).astype('int8').reset_index(name='BURO_OVERDUE_EVER'),on='SK_ID_CURR', how='left')

    del bureau; gc.collect()
    return buro_feat

buro_feat = bureau_and_balance_features()


<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">3. Previous Application table Features </h2>

In [ ]:
def previous_application_features():
    
    prev = read_csv(DATA_DIR / "previous_application.csv")
    
    # Replacing it with NaN allows algorithms like LightGBM to handle it correctly.
    for col in [c for c in prev.columns if 'DAYS_' in c]:
        prev[col] = prev[col].replace(365243, np.nan)
        
    # Ratio of what the client asked for vs. what they actually received
    prev['APP_CREDIT_RATIO']   = prev['AMT_APPLICATION'] / (prev['AMT_CREDIT'] + 1)
    # Ratio of the credit amount to the price of the goods being financed
    prev['CREDIT_GOODS_P']     = prev['AMT_CREDIT'] / (prev['AMT_GOODS_PRICE'] + 1)
    # Ratio of application amount to the price of goods
    prev['APP_GOODS_RATIO']    = prev['AMT_APPLICATION'] / (prev['AMT_GOODS_PRICE'] + 1)
    # Difference between the first due date and the first disbursement
    prev['DAYS_FIRST_DUE_DIFF']= prev['DAYS_FIRST_DUE'] - prev['DAYS_FIRST_DRAWING']
    # Difference between the scheduled end of the loan and the actual end
    prev['DAYS_LAST_DUE_DIFF'] = prev['DAYS_LAST_DUE_1ST_VERSION'] - prev['DAYS_LAST_DUE']
    # Percentage of the credit covered by the down payment
    prev['DOWN_PAYMENT_P']     = prev['AMT_DOWN_PAYMENT'] / (prev['AMT_CREDIT'] + 1)
    # Rough calculation of the total interest (Annuity * Count - Principal)
    prev['INTEREST_SHARE']     = prev['CNT_PAYMENT'] * prev['AMT_ANNUITY'] - prev['AMT_CREDIT']
    # Approximate interest rate for the application
    prev['INTEREST_RATE']      = prev['INTEREST_SHARE'] / (prev['AMT_CREDIT'] + 1)

    # Numerical Aggregations
    num_cols = [c for c in prev.columns if prev[c].dtype != 'object' and c not in ['SK_ID_CURR', 'SK_ID_PREV']]
    prev_num = prev.groupby('SK_ID_CURR')[num_cols].agg(['min', 'max', 'mean', 'sum', 'var'])
    # Flatten multi-index column names (e.g., AMT_CREDIT_MEAN)
    prev_num.columns = [f'PREV_{c[0]}_{c[1].upper()}' for c in prev_num.columns]
    prev_feat = prev_num.reset_index()

    # Calculate the frequency of each category
    cat_cols = [c for c in prev.columns if prev[c].dtype == 'object']
    if cat_cols:
        prev_cat = pd.get_dummies(prev[['SK_ID_CURR'] + cat_cols], columns=cat_cols, dummy_na=True)
        prev_cat = prev_cat.groupby('SK_ID_CURR').mean().reset_index()
        prev_cat.columns = ['SK_ID_CURR'] + [f'PREV_{c}' for c in prev_cat.columns if c != 'SK_ID_CURR']
        prev_feat = prev_feat.merge(prev_cat, on='SK_ID_CURR', how='left')
   
    # Count of previous applications per client
    prev_feat = prev_feat.merge(prev.groupby('SK_ID_CURR').size().reset_index(name='PREV_COUNT'),on='SK_ID_CURR', how='left')

    # help model to see if a client's refused applications differ from their approved ones.
    for status in ['Approved', 'Refused', 'Canceled']:
        sub = prev[prev['NAME_CONTRACT_STATUS'] == status]
        if len(sub) > 0:
            sa = sub.groupby('SK_ID_CURR')[['AMT_CREDIT', 'AMT_APPLICATION','AMT_ANNUITY', 'DAYS_DECISION']].agg(['mean', 'max', 'min'])
            sa.columns = [f'PREV_{status.upper()}_{c[0]}_{c[1].upper()}' for c in sa.columns]
            sa = sa.reset_index()
            sc = sub.groupby('SK_ID_CURR').size().reset_index(name=f'PREV_{status.upper()}_COUNT')
            prev_feat = prev_feat.merge(sa.merge(sc, on='SK_ID_CURR', how='left'),on='SK_ID_CURR', how='left')

    # Differentiating between Cash loans and Revolving loans
    for ctype in ['Cash loans', 'Revolving loans']:
        sub = prev[prev['NAME_CONTRACT_TYPE'] == ctype]
        if len(sub) > 0:
            label = 'CASH' if 'Cash' in ctype else 'REVOLV'
            sa = sub.groupby('SK_ID_CURR')[['AMT_CREDIT', 'AMT_ANNUITY','APP_CREDIT_RATIO']].agg(['mean', 'sum', 'max'])
            sa.columns = [f'PREV_{label}_{c[0]}_{c[1].upper()}' for c in sa.columns]
            sa = sa.reset_index()
            sc = sub.groupby('SK_ID_CURR').size().reset_index(name=f'PREV_{label}_COUNT')
            prev_feat = prev_feat.merge(sa.merge(sc, on='SK_ID_CURR', how='left'),  on='SK_ID_CURR', how='left')

    # Calculate stats for the last 6 months, 1 year, 2 years, etc.
    tw_cols = ['AMT_CREDIT', 'AMT_ANNUITY', 'APP_CREDIT_RATIO', 'INTEREST_RATE']
    tw_cols = [c for c in tw_cols if c in prev.columns]
    for days, label in [(-180, '6M'), (-365, '1Y'), (-730, '2Y'), (-1095, '3Y')]:
        tw = agg_time_window(prev, 'SK_ID_CURR', tw_cols, 'DAYS_DECISION', days, f'PREV_{label}')
        prev_feat = prev_feat.merge(tw, on='SK_ID_CURR', how='left')

    # Time-weighted aggregations
    tw_feats = time_weighted_agg(prev, 'SK_ID_CURR',['AMT_CREDIT', 'AMT_ANNUITY', 'APP_CREDIT_RATIO'],'DAYS_DECISION', 'PREV', decay=0.001)
    prev_feat = prev_feat.merge(tw_feats, on='SK_ID_CURR', how='left')

    # Percentage of a client's applications that were approved
    app_rate = prev.groupby('SK_ID_CURR')['NAME_CONTRACT_STATUS'].apply(lambda x: (x == 'Approved').mean()).reset_index(name='PREV_APPROVAL_RATE')
    prev_feat = prev_feat.merge(app_rate, on='SK_ID_CURR', how='left')

    # Captures the status of the client's last interaction
    prev_sorted = prev.sort_values('DAYS_DECISION', ascending=False)
    last_prev = prev_sorted.groupby('SK_ID_CURR').first().reset_index()
    for col in ['DAYS_DECISION', 'AMT_CREDIT', 'APP_CREDIT_RATIO', 'INTEREST_RATE']:
        if col in last_prev.columns:
            prev_feat = prev_feat.merge(last_prev[['SK_ID_CURR', col]].rename(columns={col: f'PREV_LAST_{col}'}),on='SK_ID_CURR', how='left')

    del prev; gc.collect()
    
    return prev_feat

prev_feat = previous_application_features()

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">4. POS Cash table Features </h2>

In [ ]:
def pos_cash_features():
   
    pos = read_csv(DATA_DIR / "POS_CASH_balance.csv")
    
    # Ratio of Days Past Due (DPD) vs Default DPD (shows severity of delinquency)
    pos['SK_DPD_RATIO'] = pos['SK_DPD'] / (pos['SK_DPD_DEF'] + 1)
    # Binary flag: Is the payment late at all?
    pos['LATE_POS'] = (pos['SK_DPD'] > 0).astype('int8')
    
    # Basic Aggregations: Min, Max, Mean, etc., per customer
    num_cols = [c for c in pos.columns if pos[c].dtype != 'object' and c not in ['SK_ID_CURR', 'SK_ID_PREV']]
    pn = pos.groupby('SK_ID_CURR')[num_cols].agg(['min', 'max', 'mean', 'sum', 'var'])
    pn.columns = [f'POS_{c[0]}_{c[1].upper()}' for c in pn.columns]
    pn = pn.reset_index()
    
    # Contract Status: % of time a customer was 'Active', 'Completed', etc.
    if 'NAME_CONTRACT_STATUS' in pos.columns:
        pc = pd.get_dummies(pos[['SK_ID_CURR', 'NAME_CONTRACT_STATUS']],columns=['NAME_CONTRACT_STATUS'], dummy_na=True)
        pc = pc.groupby('SK_ID_CURR').mean().reset_index()
        pc.columns = ['SK_ID_CURR'] + [f'POS_{c}' for c in pc.columns if c != 'SK_ID_CURR']
        pn = pn.merge(pc, on='SK_ID_CURR', how='left')
    
    # Total records  overall late-payment rate
    pn = pn.merge(pos.groupby('SK_ID_CURR').size().reset_index(name='POS_COUNT'),on='SK_ID_CURR', how='left')
    # overall late-payment rate
    pn = pn.merge(pos.groupby('SK_ID_CURR')['LATE_POS'].mean().reset_index(name='POS_LATE_RATE'),on='SK_ID_CURR', how='left')

    # Captures behavior in the last 3, 6, 12, and 24 months
    tw_cols_pos = ['SK_DPD', 'SK_DPD_DEF', 'CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE']
    for months, label in [(-3, '3M'), (-6, '6M'), (-12, '12M'), (-24, '24M')]:
        tw = agg_time_window(pos, 'SK_ID_CURR', tw_cols_pos, 'MONTHS_BALANCE',months, f'POS_{label}')
        pn = pn.merge(tw, on='SK_ID_CURR', how='left')

    # Summarizes behavior within specific loans, then aggregates to client level
    loan_agg = pos.groupby(['SK_ID_CURR', 'SK_ID_PREV']).agg(
        POS_PL_DPD_MAX=('SK_DPD', 'max'),
        POS_PL_DPD_MEAN=('SK_DPD', 'mean'),
        POS_PL_LATE_RATE=('LATE_POS', 'mean'),
        POS_PL_MONTHS=('MONTHS_BALANCE', 'count'),
    ).reset_index()
    pl_cols = [c for c in loan_agg.columns if c.startswith('POS_PL_')]
    pos_pl = loan_agg.groupby('SK_ID_CURR')[pl_cols].agg(['mean', 'max', 'std'])
    pos_pl.columns = [f'{c[0]}_{c[1].upper()}' for c in pos_pl.columns]
    pos_pl = pos_pl.reset_index()
    pn = pn.merge(pos_pl, on='SK_ID_CURR', how='left')

    # Time-weighted
    tw_feats = time_weighted_agg(pos, 'SK_ID_CURR', ['SK_DPD', 'CNT_INSTALMENT_FUTURE'],'MONTHS_BALANCE', 'POS', decay=0.02)
    pn = pn.merge(tw_feats, on='SK_ID_CURR', how='left')

    # Trend
    trend = compute_trend(pos, 'SK_ID_CURR', 'SK_DPD', 'MONTHS_BALANCE', 'POS')
    pn = pn.merge(trend, on='SK_ID_CURR', how='left')

    # What % of their total loans have they successfully finished?
    if 'NAME_CONTRACT_STATUS' in pos.columns:
        completed = pos[pos['NAME_CONTRACT_STATUS'] == 'Completed']
        comp_rate = completed.groupby('SK_ID_CURR').size() / pos.groupby('SK_ID_CURR').size()
        comp_rate = comp_rate.reset_index(name='POS_COMPLETED_RATE')
        pn = pn.merge(comp_rate, on='SK_ID_CURR', how='left')

    del pos, loan_agg; gc.collect()
    return pn

pos_feat = pos_cash_features()

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">5.  Credit Card table Features </h2>

In [ ]:
def credit_card_features():
    
    cc = read_csv(DATA_DIR / "credit_card_balance.csv")
    # How close is the customer to maxing out their card?
    cc['CC_BAL_LIM_RATIO']  = cc['AMT_BALANCE'] / (cc['AMT_CREDIT_LIMIT_ACTUAL'] + 1)
    # Is the customer paying back what they owe?
    cc['CC_PAY_TOTAL_RATIO'] = cc['AMT_PAYMENT_TOTAL_CURRENT'] / (cc['AMT_TOTAL_RECEIVABLE'] + 1)
    # Are they withdrawing cash from the ATM? (Sign of high distress)
    cc['CC_DRAW_LIM']= cc['AMT_DRAWINGS_CURRENT'] / (cc['AMT_CREDIT_LIMIT_ACTUAL'] + 1)
    # Was the customer late on this specific month?
    cc['CC_LATE']= (cc['SK_DPD'] > 0).astype('int8')
    # Paying only the minimum regularity is often a "hidden" sign of financial struggle.
    cc['CC_MIN_PAY_RATIO'] = cc['AMT_INST_MIN_REGULARITY'] / (cc['AMT_PAYMENT_CURRENT'] + 1)

    # Calculate global stats (Min, Max, Mean, Var) for all monthly records per customer
    num_cols = [c for c in cc.columns if cc[c].dtype != 'object' and c not in ['SK_ID_CURR', 'SK_ID_PREV']]
    cn = cc.groupby('SK_ID_CURR')[num_cols].agg(['min', 'max', 'mean', 'sum', 'var'])
    cn.columns = [f'CC_{c[0]}_{c[1].upper()}' for c in cn.columns]
    cn = cn.reset_index()
    # Add count of records and the overall late payment rate
    cn = cn.merge(cc.groupby('SK_ID_CURR').size().reset_index(name='CC_COUNT'),on='SK_ID_CURR', how='left')
    cn = cn.merge(cc.groupby('SK_ID_CURR')['CC_LATE'].mean().reset_index(name='CC_LATE_RATE'),on='SK_ID_CURR', how='left')

    # Behavior in the last 3M, 6M, 12M, and 24M
    tw_cols_cc = ['AMT_BALANCE', 'CC_BAL_LIM_RATIO', 'CC_DRAW_LIM', 'SK_DPD']
    for months, label in [(-3, '3M'), (-6, '6M'), (-12, '12M'), (-24, '24M')]:
        tw = agg_time_window(cc, 'SK_ID_CURR', tw_cols_cc, 'MONTHS_BALANCE',months, f'CC_{label}')
        cn = cn.merge(tw, on='SK_ID_CURR', how='left')

    # Double Aggregation (Per-Loan then Per-Client)
    loan_agg = cc.groupby(['SK_ID_CURR', 'SK_ID_PREV']).agg(
        CC_PL_BAL_LIM_MAX=('CC_BAL_LIM_RATIO', 'max'),
        CC_PL_BAL_LIM_MEAN=('CC_BAL_LIM_RATIO', 'mean'),
        CC_PL_DRAW_MEAN=('CC_DRAW_LIM', 'mean'),
        CC_PL_DPD_MAX=('SK_DPD', 'max'),
        CC_PL_LATE_RATE=('CC_LATE', 'mean'),
    ).reset_index()
    
    pl_cols = [c for c in loan_agg.columns if c.startswith('CC_PL_')]
    cc_pl = loan_agg.groupby('SK_ID_CURR')[pl_cols].agg(['mean', 'max', 'std'])
    cc_pl.columns = [f'{c[0]}_{c[1].upper()}' for c in cc_pl.columns]
    cc_pl = cc_pl.reset_index()
    cn = cn.merge(cc_pl, on='SK_ID_CURR', how='left')

    # Time-weighted
    tw_feats = time_weighted_agg(cc, 'SK_ID_CURR',['AMT_BALANCE', 'CC_BAL_LIM_RATIO', 'SK_DPD'],'MONTHS_BALANCE', 'CC', decay=0.02)
    cn = cn.merge(tw_feats, on='SK_ID_CURR', how='left')

    # Trend
    for col in ['AMT_BALANCE', 'CC_BAL_LIM_RATIO']:
        trend = compute_trend(cc, 'SK_ID_CURR', col, 'MONTHS_BALANCE', 'CC')
        cn = cn.merge(trend, on='SK_ID_CURR', how='left')

    del cc, loan_agg; gc.collect()
    return cn

cc_feat = credit_card_features()

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">6.  Installments table Features </h2>

In [ ]:
def installments_features():

    ins = read_csv(DATA_DIR / "installments_payments.csv")
    
    # Payment Percentage
    ins['PAYMENT_PERC'] = (ins['AMT_PAYMENT'] / (ins['AMT_INSTALMENT'] + 0.001)).replace([np.inf, -np.inf], np.nan).astype('float32')
    # Payment Difference
    ins['PAYMENT_DIFF'] = (ins['AMT_INSTALMENT'] - ins['AMT_PAYMENT']).astype('float32')
    # How many days late (if payment was after the deadline)
    ins['DPD']  = np.maximum(ins['DAYS_ENTRY_PAYMENT'] - ins['DAYS_INSTALMENT'], 0).astype('float32')
    # How many days early (if payment was before the deadline - a very good sign)
    ins['DBD']  = np.maximum(ins['DAYS_INSTALMENT'] - ins['DAYS_ENTRY_PAYMENT'], 0).astype('float32')
   
    # Binary Flags for the model
    ins['LATE_PAYMENT'] = (ins['DPD'] > 0).astype('int8')
    ins['SIGNIFICANT_UNDERPAY'] = (ins['PAYMENT_DIFF'] > 100).astype('int8')
   
    # Global Aggregations per customer (SK_ID_CURR)
    num_cols = [c for c in ins.columns if ins[c].dtype != 'object' and c not in ['SK_ID_CURR', 'SK_ID_PREV']]
   
    iN = ins.groupby('SK_ID_CURR')[num_cols].agg(['min', 'max', 'mean', 'sum', 'var'])
    iN.columns = [f'INS_{c[0]}_{c[1].upper()}' for c in iN.columns]
    iN = iN.reset_index()
    
    # Add overall late rates and underpayment rates
    iN = iN.merge(ins.groupby('SK_ID_CURR').size().reset_index(name='INS_COUNT'),on='SK_ID_CURR', how='left')
    iN = iN.merge(ins.groupby('SK_ID_CURR')['LATE_PAYMENT'].mean().reset_index(name='INS_LATE_RATE'),on='SK_ID_CURR', how='left')
    iN = iN.merge(ins.groupby('SK_ID_CURR')['SIGNIFICANT_UNDERPAY'].mean().reset_index(name='INS_SIGUNDERPAY_RATE'), on='SK_ID_CURR', how='left')

    # Focus on activity in the last 6 months, 1 year, and 2 years
    tw_cols_ins = ['DPD', 'PAYMENT_PERC', 'PAYMENT_DIFF', 'LATE_PAYMENT']
    for days, label in [(-180, '6M'), (-365, '1Y'), (-730, '2Y')]:
        tw = agg_time_window(ins, 'SK_ID_CURR', tw_cols_ins, 'DAYS_INSTALMENT',days, f'INS_{label}')
        iN = iN.merge(tw, on='SK_ID_CURR', how='left')

    # Summarize each loan's behavior first, then calculate the client's average behavior across loans
    loan_agg = ins.groupby(['SK_ID_CURR', 'SK_ID_PREV']).agg(
        INS_PL_DPD_MEAN=('DPD', 'mean'),
        INS_PL_DPD_MAX=('DPD', 'max'),
        INS_PL_LATE_SUM=('LATE_PAYMENT', 'sum'),
        INS_PL_LATE_RATE=('LATE_PAYMENT', 'mean'),
        INS_PL_PAYPERC_MEAN=('PAYMENT_PERC', 'mean'),
        INS_PL_PAYPERC_MIN=('PAYMENT_PERC', 'min'),
        INS_PL_PAYDIFF_MAX=('PAYMENT_DIFF', 'max'),
        INS_PL_COUNT=('DPD', 'size'),
    ).reset_index()
    
    pl_cols = [c for c in loan_agg.columns if c.startswith('INS_PL_')]
    ins_pl = loan_agg.groupby('SK_ID_CURR')[pl_cols].agg(['mean', 'max', 'std'])
    ins_pl.columns = [f'{c[0]}_{c[1].upper()}' for c in ins_pl.columns]
    ins_pl = ins_pl.reset_index()
    iN = iN.merge(ins_pl, on='SK_ID_CURR', how='left')

    # Time-weighted
    tw_feats = time_weighted_agg(ins, 'SK_ID_CURR',['DPD', 'PAYMENT_PERC', 'PAYMENT_DIFF'],'DAYS_INSTALMENT', 'INS', decay=0.001)
    iN = iN.merge(tw_feats, on='SK_ID_CURR', how='left')

    # Trend
    for col in ['DPD', 'PAYMENT_PERC']:
        trend = compute_trend(ins, 'SK_ID_CURR', col, 'DAYS_INSTALMENT', 'INS')
        iN = iN.merge(trend, on='SK_ID_CURR', how='left')

    # If a client was perfect for 2 years but missed the last 3 payments, 
    # they are currently high-risk. This isolates that recency.
    ins_sorted = ins.sort_values('DAYS_INSTALMENT', ascending=False)
    for k in [3, 5, 10, 30]:
        last_k = ins_sorted.groupby('SK_ID_CURR').head(k)
        lk_agg = last_k.groupby('SK_ID_CURR').agg(
            **{f'INS_LAST{k}_DPD_MEAN': ('DPD', 'mean'),
               f'INS_LAST{k}_DPD_MAX': ('DPD', 'max'),
               f'INS_LAST{k}_PAYPERC_MEAN': ('PAYMENT_PERC', 'mean'),
               f'INS_LAST{k}_PAYDIFF_MEAN': ('PAYMENT_DIFF', 'mean'),
               f'INS_LAST{k}_LATE_RATE': ('LATE_PAYMENT', 'mean')}
        ).reset_index()
        iN = iN.merge(lk_agg, on='SK_ID_CURR', how='left')

    # Changes in version often mean loan re-scheduling
    if 'NUM_INSTALMENT_VERSION' in ins.columns:
        ver_agg = ins.groupby('SK_ID_CURR')['NUM_INSTALMENT_VERSION'].agg(
            INS_VERSION_NUNIQUE='nunique',
            INS_VERSION_MAX='max',
            INS_VERSION_MEAN='mean'
        ).reset_index()
        iN = iN.merge(ver_agg, on='SK_ID_CURR', how='left')

    del ins, loan_agg, ins_sorted; gc.collect()
    return iN
    
ins_feat = installments_features()

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">7.  Sub-Model Features </h2>

##### Trains a small model on each dataset and uses its predictions as smart features for the final model.
##### Rich Behavioral Data
- `bureau` → credit history  
- `previous_application` → past loan behavior  
- `installments_payments` → repayment behavior  

They capture **real user behavior**, which strongly relates to the target.

---
- Each customer has multiple records  
 Perfect for:
- Row-level modeling  
- Then aggregation → strong features  

In [ ]:
def sub_model_features_fixed(table_path, target_df, prefix):
    df = read_csv(table_path)
    
    # Replace abnormal placeholder values in DAYS_ columns
    for col in df.columns:
        if 'DAYS_' in col:
            df[col] = df[col].replace(365243, np.nan)
            
    # Select only numeric features (exclude IDs and categorical columns)
    feat_cols = [c for c in df.columns if df[c].dtype != 'object'and c not in ['SK_ID_CURR', 'SK_ID_PREV', 'SK_ID_BUREAU']]

    # Identify which rows belong to training set (have TARGET)
    train_ids_set = set(target_df['SK_ID_CURR'].values)
    mask_train = df['SK_ID_CURR'].isin(train_ids_set)
    
    # Split into train rows and test rows
    df_train_rows = df[mask_train].copy()
    df_test_rows  = df[~mask_train].copy()
    
    # Attach TARGET to training rows
    df_train_rows = df_train_rows.merge(target_df[['SK_ID_CURR', 'TARGET']],on='SK_ID_CURR', how='inner')
    
    # Prepare training data (handle inf values)
    X_train_all = df_train_rows[feat_cols].replace([np.inf, -np.inf], np.nan)
    y_train_all = df_train_rows['TARGET'].astype(int)

    # Group by customer to avoid leakage across folds
    groups_train = df_train_rows['SK_ID_CURR']
    
    # Store out-of-fold predictions
    oof_preds = np.zeros(len(df_train_rows))
    fitted_models = []

    # GroupKFold ensures same customer doesn't appear in both train/valid
    gkf = GroupKFold(n_splits=5)
    
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train_all, y_train_all, groups=groups_train)):
        
        # Train LightGBM model
        
        m = lgb.LGBMClassifier(
            n_estimators=2000, learning_rate=0.05, num_leaves=31, max_depth=5,
            subsample=0.8, colsample_bytree=0.5, reg_alpha=0.1, reg_lambda=0.1,
            min_child_samples=100, random_state=SEED, n_jobs=-1, verbose=-1)
        
        # Fit with early stopping
        
        m.fit(X_train_all.iloc[tr_idx], y_train_all.iloc[tr_idx],
              eval_set=[(X_train_all.iloc[va_idx], y_train_all.iloc[va_idx])],
              eval_metric='auc',
              callbacks=[lgb.early_stopping(50, verbose=False)])
        
        oof_preds[va_idx] = m.predict_proba(X_train_all.iloc[va_idx])[:, 1]
        
        # Save out-of-fold predictions (no leakage)
        
        fitted_models.append(m)
        
    # Evaluate sub-model performance
    auc = roc_auc_score(y_train_all, oof_preds)
    print(f"  {prefix} sub-model OOF AUC: {auc:.4f}")
  
    # Assign OOF predictions to training rows
    df_train_rows['_SUB_PRED'] = oof_preds
   
    # Prepare test data
    X_test_rows = df_test_rows[feat_cols].replace([np.inf, -np.inf], np.nan)
    
    # Predict test rows using average of all fold models
    test_preds = np.zeros(len(df_test_rows))
    
    for m in fitted_models:
        test_preds += m.predict_proba(X_test_rows)[:, 1] / len(fitted_models)
    
    df_test_rows['_SUB_PRED'] = test_preds
    
    # Combine train + test predictions
    all_rows = pd.concat([
        df_train_rows[['SK_ID_CURR', '_SUB_PRED']],
        df_test_rows[['SK_ID_CURR', '_SUB_PRED']]
    ], axis=0)

    # Aggregate predictions per customer (meta-features)
    sub_agg = all_rows.groupby('SK_ID_CURR')['_SUB_PRED'].agg(
        **{f'{prefix}_SUB_MEAN': 'mean', f'{prefix}_SUB_MAX': 'max',
           f'{prefix}_SUB_MIN': 'min', f'{prefix}_SUB_STD': 'std'}
    ).reset_index()
    
    
    # Count number of "high-risk" rows per customer
    high_risk = (all_rows[all_rows['_SUB_PRED'] > 0.15].groupby('SK_ID_CURR').size().reset_index(name=f'{prefix}_SUB_HIGHRISK'))
    
    # Merge high-risk count
    sub_agg = sub_agg.merge(high_risk, on='SK_ID_CURR', how='left')
    sub_agg[f'{prefix}_SUB_HIGHRISK'] = sub_agg[f'{prefix}_SUB_HIGHRISK'].fillna(0)

    del df, df_train_rows, df_test_rows, all_rows; gc.collect()
    return sub_agg

target_df = app_train[['SK_ID_CURR', 'TARGET']].copy()

print("Training sub-model on previous_application rows...")
prev_sub = sub_model_features_fixed(DATA_DIR / "previous_application.csv", target_df, 'PREV')

print("Training sub-model on bureau rows...")
buro_sub = sub_model_features_fixed(DATA_DIR / "bureau.csv", target_df, 'BURO')

print("Training sub-model on installments rows...")
ins_sub = sub_model_features_fixed(DATA_DIR / "installments_payments.csv", target_df, 'INS')

print("Sub-model features done.")


<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">8.  Merge All Features </h2>

In [ ]:
feats = [buro_feat, prev_feat, pos_feat, cc_feat, ins_feat, prev_sub, buro_sub, ins_sub]

train = app_train.copy()
test  = app_test.copy()

for f in feats:
    train = train.merge(f, on='SK_ID_CURR', how='left')
    test  = test.merge(f, on='SK_ID_CURR', how='left')
    
print(f"Merged train: {train.shape}, test: {test.shape}")

del app_train, app_test, buro_feat, prev_feat, pos_feat, cc_feat, ins_feat
del prev_sub, buro_sub, ins_sub

gc.collect()


<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">9.  Adding frequency , groupby , target-encoding features </h2>

In [ ]:
print("Adding frequency / groupby / target-encoding features...")

GROUPBY_CAT_COLS = ['NAME_EDUCATION_TYPE', 'ORGANIZATION_TYPE', 'OCCUPATION_TYPE','NAME_INCOME_TYPE', 'CODE_GENDER', 'AGE_RANGE']

GROUPBY_NUM_COLS = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY','EXT_MEAN', 'CREDIT_ANNUITY_RATIO', 'ANNUITY_INCOME_RATIO','DAYS_EMPLOYED_YRS']

#  Convert categorical columns into their frequency in the dataset
cat_cols_for_freq = [c for c in train.columns if train[c].dtype == 'object']
train, test = add_frequency_features(train, test, cat_cols_for_freq)

# Create interaction features between categorical columns
combo_candidates = [
    ('NAME_EDUCATION_TYPE', 'NAME_INCOME_TYPE'),
    ('CODE_GENDER', 'NAME_FAMILY_STATUS'),
    ('OCCUPATION_TYPE', 'ORGANIZATION_TYPE'),
    ('AGE_RANGE', 'NAME_EDUCATION_TYPE')
]
for c1, c2 in combo_candidates:
    if c1 in train.columns and c2 in train.columns:
        new_col = f'{c1}__{c2}'
        # Combine two categorical values into one feature
        train[new_col] = train[c1].astype(str) + '__' + train[c2].astype(str)
        test[new_col] = test[c1].astype(str) + '__' + test[c2].astype(str)

# Groupby features
gp_cat_cols = [c for c in GROUPBY_CAT_COLS if c in train.columns]
gp_num_cols = [c for c in GROUPBY_NUM_COLS if c in train.columns]
train, test = add_groupby_ratio_features(train, test, gp_cat_cols, gp_num_cols)

# OOF Target encoding (single pass, no duplication)
te_cols = [c for c in [
    'NAME_EDUCATION_TYPE', 'ORGANIZATION_TYPE', 'OCCUPATION_TYPE',
    'NAME_INCOME_TYPE', 'CODE_GENDER', 'NAME_HOUSING_TYPE',
    'AGE_RANGE', 'NAME_EDUCATION_TYPE__NAME_INCOME_TYPE',
    'CODE_GENDER__NAME_FAMILY_STATUS', 'OCCUPATION_TYPE__ORGANIZATION_TYPE',
    'AGE_RANGE__NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS', 'NAME_CONTRACT_TYPE',
] if c in train.columns]

train, test = add_target_encoding(train, test, 'TARGET', te_cols,n_splits=N_FOLDS, smoothing=TE_SMOOTHING,min_samples_leaf=TE_MIN_SAMPLES, seed=SEED)

train = reduce_memory_usage(train)
test = reduce_memory_usage(test)
print(f"After feature block: train {train.shape}, test {test.shape}")


<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">10.  KNN Target Features </h2>

+ This code uses a **K-Nearest Neighbors (KNN) model not for final prediction**, but to generate **new features** for the main machine learning model.
+ Measure how similar each customer is to other customers and convert that similarity into a risk score.

In [ ]:
target = train['TARGET'].astype('int8')

print("Computing KNN target features...")

# We use only strong predictive continuous features
knn_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'CREDIT_ANNUITY_RATIO']

# Combine train + test for consistent scaling NOT using TARGET here → safe
X_knn_all = pd.concat([train[knn_cols], test[knn_cols]], axis=0).fillna(-999).values

# fit_transform on combined data is usually OK for scaling
scaler_knn = StandardScaler()
X_knn_all = scaler_knn.fit_transform(X_knn_all)

# Split back into train and test
X_knn_tr = X_knn_all[:len(train)]
X_knn_te = X_knn_all[len(train):]

# StratifiedKFold ensures class balance in each fold
skf_knn = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Train KNN models for different k values
for k in [200, 500]:
    
    oof_k = np.zeros(len(train))
    test_k = np.zeros(len(test))

    # Cross-validated training
    for fold, (tr_idx, va_idx) in enumerate(skf_knn.split(X_knn_tr, target)):

        # KNN model
        knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean', n_jobs=-1)
        knn.fit(X_knn_tr[tr_idx], target.values[tr_idx])

        # Predict validation fold 
        oof_k[va_idx] = knn.predict_proba(X_knn_tr[va_idx])[:, 1]
       
        # Predict test set and average over folds
        test_k += knn.predict_proba(X_knn_te)[:, 1] / 5
        
    # Save features
    train[f'KNN_TARGET_{k}'] = oof_k.astype('float32')
    test[f'KNN_TARGET_{k}']  = test_k.astype('float32')
    print(f"  KNN k={k} OOF AUC: {roc_auc_score(target, oof_k):.6f}")

del X_knn_all, X_knn_tr, X_knn_te; gc.collect()

print(f"After KNN: train {train.shape}, test {test.shape}")


<h1 style="margin: 0; padding: 20px; background-color: #2a3128; color: white; text-align: left;">5. Feature Selection </h1>

In [ ]:
# Save IDs for submission
train_ids = train['SK_ID_CURR'].copy()
test_ids  = test['SK_ID_CURR'].copy()

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">1. Prepare CatBoost Feature </h2>

In [ ]:
# Prepare CatBoost version (keeps categorical as strings)

cat_train = train.drop(columns=['SK_ID_CURR']).copy()
cat_test  = test.drop(columns=['SK_ID_CURR']).copy()
cat_feature_names = [c for c in cat_train.columns if c != 'TARGET' and cat_train[c].dtype == 'object']
for col in cat_feature_names:
    cat_train[col] = cat_train[col].fillna('__nan__').astype(str)
    cat_test[col] = cat_test[col].fillna('__nan__').astype(str)

cat_train = cat_train.replace([np.inf, -np.inf], np.nan)
cat_test  = cat_test.replace([np.inf, -np.inf], np.nan)

# Align train and test columns (keep only common columns)
cat_train_features = cat_train.drop(columns=['TARGET']).copy()
cat_train_features, cat_test = cat_train_features.align(cat_test, join='inner', axis=1)
cat_train = pd.concat([cat_train[['TARGET']].reset_index(drop=True),cat_train_features.reset_index(drop=True)], axis=1)

cat_feature_names = [c for c in cat_feature_names if c in cat_train.columns]

# Get indices of categorical features for CatBoost
cat_feature_indices = [cat_train.columns.get_loc(c) for c in cat_feature_names if c in cat_train.columns]

# Convert non-categorical columns to numeric
for col in cat_train.columns:
    if col not in cat_feature_names and col != 'TARGET':
        cat_train[col] = pd.to_numeric(cat_train[col], errors='coerce')
        cat_test[col] = pd.to_numeric(cat_test[col], errors='coerce')

# Clean column names (remove special characters)
cat_clean_names = {c: re.sub(r'[^A-Za-z0-9_]+', '_', c) for c in cat_train.columns}
cat_train = cat_train.rename(columns=cat_clean_names)
cat_test  = cat_test.rename(columns=cat_clean_names)

# Update categorical feature names after renaming
cat_feature_names = [cat_clean_names.get(c, c) for c in cat_feature_names]

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">2. Prepare LGB/XGB Feature </h2>

In [ ]:
# Numeric pipeline for LGB/XGB
train = train.drop(columns=['TARGET', 'SK_ID_CURR'])
test  = test.drop(columns=['SK_ID_CURR'])

# Label Encoding for categorical features (convert strings → integers)

obj_cols = [c for c in train.columns if train[c].dtype == 'object']
for col in obj_cols:
    le = LabelEncoder()
    all_vals = pd.concat([train[col], test[col]], axis=0).astype(str).fillna('nan')
    le.fit(all_vals)
    train[col] = le.transform(train[col].astype(str).fillna('nan')).astype('int32')
    test[col]  = le.transform(test[col].astype(str).fillna('nan')).astype('int32')

train = train.replace([np.inf, -np.inf], np.nan)
test  = test.replace([np.inf, -np.inf], np.nan)

# Align train and test columns (keep only common columns)
train, test = train.align(test, join='inner', axis=1)

# Clean column names (remove special characters)
clean_names = {c: re.sub(r'[^A-Za-z0-9_]+', '_', c) for c in train.columns}
train = train.rename(columns=clean_names)
test  = test.rename(columns=clean_names)

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">3. Drop useless columns</h2>

In [ ]:
# Drop all NaN or zero variance
drop_cols = [c for c in train.columns if train[c].isna().all() or train[c].nunique(dropna=False) <= 1]

if drop_cols:
    train = train.drop(columns=drop_cols)
    test  = test.drop(columns=drop_cols)
    cat_drop = [c for c in drop_cols if c in cat_train.columns]
    if cat_drop:
        cat_train = cat_train.drop(columns=cat_drop)
        cat_test = cat_test.drop(columns=cat_drop)

# Drop highly correlated (>0.985)
sample = train.sample(min(30000, len(train)), random_state=42)

corr = sample.corr(numeric_only=True).abs()
# Keep only the upper triangle of the correlation matrix (excluding diagonal)
# - np.ones(corr.shape): create matrix of 1s same shape as corr
# - np.triu(..., k=1): keep only upper triangle (k=1 removes diagonal)
# - astype(bool): convert to True/False mask
# - corr.where(...): keep values where mask is True, others become NaN
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

high_corr = [col for col in upper.columns if any(upper[col] > 0.985)]
if high_corr:
    train = train.drop(columns=high_corr)
    test  = test.drop(columns=high_corr)
    cat_drop = [c for c in high_corr if c in cat_train.columns]
    if cat_drop:
        cat_train = cat_train.drop(columns=cat_drop)
        cat_test = cat_test.drop(columns=cat_drop)
    cat_feature_names = [c for c in cat_feature_names if c in cat_train.columns]

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">4. CatBoost final alignment</h2>

In [ ]:
cat_y = cat_train['TARGET'].astype('int8').copy()
cat_train = cat_train.drop(columns=['TARGET'])
cat_train, cat_test = cat_train.align(cat_test, join='inner', axis=1)
cat_feature_names = [c for c in cat_feature_names if c in cat_train.columns]
cat_feature_indices = [cat_train.columns.get_loc(c) for c in cat_feature_names]

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">5. Null-Importance Feature Selection</h2>

In [ ]:
# compute feature importances
def get_importances(X, y, shuffle=False, seed=0):
    
    if shuffle:
        y = y.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    fold_imp = np.zeros(X.shape[1])
    
    # Stratified K-Fold for stability
    skf_ni = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    
    for ti, vi in skf_ni.split(X, y):
        
        # LightGBM model
        m = lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=40, max_depth=5,
            subsample=0.8, colsample_bytree=0.3, min_child_samples=50,
            random_state=seed, n_jobs=-1, verbose=-1)
        
        # Train model
        m.fit(X.iloc[ti], y.iloc[ti],
              eval_set=[(X.iloc[vi], y.iloc[vi])],
              eval_metric='auc',
              callbacks=[lgb.early_stopping(20, verbose=False)])
        
        fold_imp += m.feature_importances_
    
    return fold_imp / 3

In [ ]:
# Get feature names list
feat_names = list(train.columns)

# Sample data for faster computation
NI_SAMPLE = min(60000, len(train))
ni_idx = train.sample(NI_SAMPLE, random_state=42).index

# Subset of data
X_ni = train.loc[ni_idx].reset_index(drop=True)
y_ni = target.loc[ni_idx].reset_index(drop=True)

# Compute actual importance 
actual_imp = get_importances(X_ni, y_ni, shuffle=False, seed=42)

# Compute null importance 
N_NULL_RUNS = 5  
null_imps = np.zeros((X_ni.shape[1], N_NULL_RUNS))

for i in range(N_NULL_RUNS):
    null_imps[:, i] = get_importances(X_ni, y_ni, shuffle=True, seed=100+i)

# 80th percentile of null importance 
null_80 = np.percentile(null_imps, 80, axis=1)

# Score = how much better than noise
score_vs_null = actual_imp / (null_80 + 1)

# Only drop features clearly below noise 
# This is less aggressive to avoid removing useful features
drop_null = [feat_names[j] for j in range(len(feat_names)) if score_vs_null[j] < 1.0]


if drop_null:
    train = train.drop(columns=drop_null)
    test  = test.drop(columns=drop_null)
    feat_names = list(train.columns)

del X_ni, y_ni; gc.collect()

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">5. Feature Subsets for Model Diversity</h2>

In [ ]:
# Define LightGBM model for stable feature importance estimation

m_imp = lgb.LGBMClassifier(
    n_estimators=2000, learning_rate=0.02, num_leaves=48, max_depth=6,
    subsample=0.8, colsample_bytree=0.3, min_child_samples=50,
    random_state=42, n_jobs=-1, verbose=-1)

# Stratified K-Fold for stable importance estimation
skf_imp = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

imp_arr = np.zeros(len(feat_names))

for ti, vi in skf_imp.split(train, target):
   
    # Train model on training fold
    m_imp.fit(train.iloc[ti], target.iloc[ti],
              eval_set=[(train.iloc[vi], target.iloc[vi])],
              eval_metric='auc',
              callbacks=[lgb.early_stopping(50, verbose=False)])
    
    imp_arr += m_imp.feature_importances_

imp_arr /= 3

# Rank features by importance
imp_rank = pd.DataFrame({'feature': feat_names, 'imp': imp_arr}).sort_values('imp', ascending=False)

# All features (baseline)
feat_all = feat_names
# Top 400 most important features
feat_top400 = imp_rank.head(min(400, len(feat_names)))['feature'].tolist()
# Features excluding engineered GP features --> gplearn
feat_no_gp = [f for f in feat_names if not f.startswith('GP')]

del m_imp; gc.collect()


<h1 style="margin: 0; padding: 20px; background-color: #2a3128; color: white; text-align: left;">5. Model </h1>

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">Model 1: LGB Config A</h2>

In [ ]:
# Stratified K-Fold 
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
# Array to store Out-Of-Fold predictions for training data
oof_lgb1 = np.zeros(len(train))
# Array to store averaged predictions for test data
test_lgb1 = np.zeros(len(test))

# LightGBM Parameters 
lgb_params_a = dict(
    objective='binary', metric='auc', boosting_type='gbdt',
    n_estimators=10000, learning_rate=0.01, num_leaves=48, max_depth=7,
    subsample=0.8, colsample_bytree=0.25, reg_alpha=0.05, reg_lambda=0.1,
    min_child_samples=40, min_child_weight=30,
    random_state=42, n_jobs=-1, verbose=-1
)

# Cross Validation Training Loop
for fold, (ti, vi) in enumerate(skf.split(train, target)):
 
    m = lgb.LGBMClassifier(**lgb_params_a)
    
    m.fit(train.iloc[ti], target.iloc[ti],
          eval_set=[(train.iloc[vi], target.iloc[vi])],
          eval_metric='auc',
          # Early stopping to prevent overfitting
          callbacks=[lgb.early_stopping(200, verbose=True), lgb.log_evaluation(500)])
    
    # Predict probabilities for validation set
    oof_lgb1[vi] = m.predict_proba(train.iloc[vi])[:, 1]
    # Predict on test set and average results
    test_lgb1 += m.predict_proba(test)[:, 1] / N_FOLDS

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">Model 2: LGB Config B (different seed + top features)</h2>

In [ ]:
# Second CV setup (different random seed for diversity)
skf2 = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=123)

# OOF predictions for model B
oof_lgb2 = np.zeros(len(train))
# Test predictions for model B
test_lgb2 = np.zeros(len(test))

# Use Top 400 features only
train_b = train[feat_top400]
test_b  = test[feat_top400]

# LightGBM parameters (different configuration for diversity)
lgb_params_b = dict(
    objective='binary', metric='auc', boosting_type='gbdt',
    n_estimators=10000, learning_rate=0.008, num_leaves=34, max_depth=5,
    subsample=0.75, colsample_bytree=0.35, reg_alpha=0.05, reg_lambda=0.2,
    min_child_samples=60, min_child_weight=50,
    random_state=123, n_jobs=-1, verbose=-1
)

# Cross Validation Training Loop
for fold, (ti, vi) in enumerate(skf2.split(train_b, target)):
    
    # Initialize model
    m = lgb.LGBMClassifier(**lgb_params_b)

    # Train model
    m.fit(train_b.iloc[ti], target.iloc[ti],
          eval_set=[(train_b.iloc[vi], target.iloc[vi])],
          eval_metric='auc',
          callbacks=[lgb.early_stopping(200, verbose=True), lgb.log_evaluation(500)])
    # OOF Predictions
    oof_lgb2[vi] = m.predict_proba(train_b.iloc[vi])[:, 1]
    # Test Predictions
    test_lgb2 += m.predict_proba(test_b)[:, 1] / N_FOLDS
   
del train_b, test_b; gc.collect()


<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">Model 3: LGB Seed Averaging</h2>

In [ ]:
# OOF predictions 
oof_lgb3 = np.zeros(len(train))
# Test predictions 
test_lgb3 = np.zeros(len(test))

# Use features excluding GP (more stable set)
train_c = train[feat_no_gp]
test_c  = test[feat_no_gp]

# Number of different random seeds
N_SEEDS = 3

for seed_i, seed in enumerate([456, 789, 1234]):
    
    skf3 = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    
    # Store predictions for this seed
    oof_seed = np.zeros(len(train))
    test_seed = np.zeros(len(test))

    # Cross Validation loop
    for fold, (ti, vi) in enumerate(skf3.split(train_c, target)):
       
        # Initialize model (same config, different seed)
        m = lgb.LGBMClassifier(
            objective='binary', metric='auc', boosting_type='gbdt',
            n_estimators=10000, learning_rate=0.01, num_leaves=40, max_depth=6,
            subsample=0.8, colsample_bytree=0.3, reg_alpha=0.1, reg_lambda=0.15,
            min_child_samples=50, min_child_weight=40,
            random_state=seed, n_jobs=-1, verbose=-1)
       
        # Train model
        m.fit(train_c.iloc[ti], target.iloc[ti],
              eval_set=[(train_c.iloc[vi], target.iloc[vi])],
              eval_metric='auc',
              callbacks=[lgb.early_stopping(200, verbose=False)])
        
        # OOF predictions for this seed
        oof_seed[vi] = m.predict_proba(train_c.iloc[vi])[:, 1]
        # Test predictions (averaged across folds)
        test_seed += m.predict_proba(test_c)[:, 1] / N_FOLDS
     
    # Aggregate across seeds
    oof_lgb3 += oof_seed / N_SEEDS
    test_lgb3 += test_seed / N_SEEDS

del train_c, test_c; gc.collect()

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">Model 4: XGBoost</h2>

In [ ]:
# Initialize OOF and test predictions
oof_xgb = np.zeros(len(train))
test_xgb = np.zeros(len(test))

# Handle XGBoost version differences (GPU vs CPU)
xgb_version = tuple(int(x) for x in xgb.__version__.split('.')[:2])
if xgb_version >= (2, 0):
    xgb_extra = {'device': 'cuda', 'tree_method': 'hist'} if HAS_GPU else {'tree_method': 'hist'}
else:
    xgb_extra = {'tree_method': 'gpu_hist'} if HAS_GPU else {'tree_method': 'hist'}

# Cross Validation Training Loop
for fold, (ti, vi) in enumerate(skf.split(train, target)):
   
    # Initialize XGBoost model
    m = xgb.XGBClassifier(objective='binary:logistic', eval_metric='auc',
        n_estimators=10000, learning_rate=0.01, max_depth=5,
        subsample=0.8, colsample_bytree=0.3, reg_alpha=0.1, reg_lambda=1.0,
        min_child_weight=40, gamma=0.1,  
        random_state=42, n_jobs=-1, verbosity=0,
        early_stopping_rounds=200, **xgb_extra)
    
    # Train model
    m.fit(train.iloc[ti], target.iloc[ti],
          eval_set=[(train.iloc[vi], target.iloc[vi])], verbose=500)
    
    # OOF Predictions
    oof_xgb[vi] = m.predict_proba(train.iloc[vi])[:, 1]
    # Test Predictions (averaged across folds)
    test_xgb += m.predict_proba(test)[:, 1] / N_FOLDS

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">Model 5: CatBoost</h2>

In [ ]:
# Initialize OOF and test predictions
oof_cat = np.zeros(len(cat_train))
test_cat = np.zeros(len(cat_test))

# Select device (GPU if available)
cat_task = 'GPU' if HAS_GPU else 'CPU'

# Cross Validation Training Loop
for fold, (ti, vi) in enumerate(skf.split(cat_train, cat_y)):
    
    # CatBoost parameters
    cat_params = dict(
        loss_function='Logloss',
        eval_metric='AUC',
        iterations=10000,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=3.0,  
        random_seed=42 + fold,
        verbose=500,
        early_stopping_rounds=300,
        task_type=cat_task,
        bootstrap_type='Bernoulli',
        subsample=0.8,
        grow_policy='SymmetricTree',
        leaf_estimation_iterations=3,
    )

    # CPU-specific parameter (feature sampling)
    if cat_task == 'CPU':
        cat_params['rsm'] = 0.3
        
    # Create CatBoost Pools (optimized data format)
    train_pool = Pool(cat_train.iloc[ti], label=cat_y.iloc[ti], cat_features=cat_feature_indices)
    valid_pool = Pool(cat_train.iloc[vi], label=cat_y.iloc[vi], cat_features=cat_feature_indices)
    test_pool  = Pool(cat_test, cat_features=cat_feature_indices)

    m = CatBoostClassifier(**cat_params)
    
    m.fit(train_pool, eval_set=valid_pool, use_best_model=True)
    
    # OOF Predictions
    oof_cat[vi] = m.predict_proba(valid_pool)[:, 1]
    # test Predictions
    test_cat += m.predict_proba(test_pool)[:, 1] / N_FOLDS
    

<h2 style="margin: 0; padding: 20px; background-color: #DC143C; color: white; text-align: left;">Averaging Ensemble</h2>

##### 1. Stacking

* Train multiple base models
* Use their predictions as input features
* Train a meta-model (e.g., Logistic Regression)

Flow:
Base Models → Predictions → Meta Model → Final Prediction

---

##### 2. Weighted Blending

* Take predictions from all models
* Combine them using weights
* Optimize weights to maximize AUC

Formula:
Final = w1*M1 + w2*M2 + ... + wn*Mn

In [ ]:
# Rank normalization (convert predictions to ranks)
def rank_norm(a):
    return rankdata(a) / len(a)

# Collect all models (OOF + test predictions)
models = {
    'lgb_a':      (oof_lgb1, test_lgb1),
    'lgb_b':      (oof_lgb2, test_lgb2),
    'lgb_seed':   (oof_lgb3, test_lgb3),
    'xgb':        (oof_xgb,  test_xgb),
    'cat':        (oof_cat,  test_cat),
}

print("=== Individual Model OOF AUCs ===")
for name, (oof, _) in models.items():
    print(f"  {name}: {roc_auc_score(target, oof):.6f}")

# Used to measure diversity (lower correlation = better ensemble)
print("\n=== Pairwise Rank Correlations ===")
names = list(models.keys())
for i in range(len(names)):
    for j in range(i+1, len(names)):
        r = np.corrcoef(rank_norm(models[names[i]][0]),rank_norm(models[names[j]][0]))[0, 1]
        print(f"  {names[i]:12s} vs {names[j]:12s}: {r:.4f}")

# Level-2 Logistic Stacker
print("\n=== Level-2 Stacking ===")
# Optional strong raw features to include in stacking
raw_stack_cols = []
for c in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3','CREDIT_ANNUITY_RATIO', 'KNN_TARGET_500', 'ANNUITY_INCOME_RATIO']:
    
    if c in train.columns:
        raw_stack_cols.append(c)
        
# Stack model predictions as features
oof_stack = np.column_stack([models[n][0] for n in names])
test_stack = np.column_stack([models[n][1] for n in names])

# append raw features to stacking input
if raw_stack_cols:
    raw_tr = train[raw_stack_cols].fillna(-999).values.astype('float32')
    raw_te = test[raw_stack_cols].fillna(-999).values.astype('float32')
    
    oof_stack = np.column_stack([oof_stack, raw_tr])
    test_stack = np.column_stack([test_stack, raw_te])

# Train Logistic Regression as meta-model
oof_stack_lr = np.zeros(len(train))
test_stack_lr = np.zeros(len(test))

skf_stack = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=789)

for fold, (ti, vi) in enumerate(skf_stack.split(oof_stack, target)):
    # Simple but effective stacking model
    lr = LogisticRegression(C=0.35, max_iter=2000, solver='lbfgs', random_state=789)
    # Train on stacked features
    lr.fit(oof_stack[ti], target.values[ti])
    
    oof_stack_lr[vi] = lr.predict_proba(oof_stack[vi])[:, 1]
    test_stack_lr += lr.predict_proba(test_stack)[:, 1] / N_FOLDS

stack_lr_auc = roc_auc_score(target, oof_stack_lr)
print(f"  Logistic stacked OOF AUC: {stack_lr_auc:.6f}")

# Rank-normalized predictions for all models + stacker
print("\n=== Constrained Rank-Blending ===")
blend_oof = [rank_norm(models[n][0]) for n in names] + [rank_norm(oof_stack_lr)]
blend_test = [rank_norm(models[n][1]) for n in names] + [rank_norm(test_stack_lr)]
blend_names = names + ['stack_lr']

def neg_auc(w):
    w = np.clip(w, 0, 1)
    w = w / (w.sum() + 1e-12)
    blend = sum(wi * ri for wi, ri in zip(w, blend_oof))
    return -roc_auc_score(target, blend)

# Initial weights (uniform)
n_models = len(blend_names)
w0 = np.ones(n_models) / n_models
# Weight constraints
bounds = [(0.0, 0.60)] * n_models
# Constraint: sum of weights = 1
cons = ({'type': 'eq', 
         'fun': lambda w: np.sum(np.clip(w, 0, 1)) - 1.0},)

# Optimize blending weights
best_result = None
for method in ['SLSQP', 'Powell']:
    try:
        result = minimize(neg_auc, w0, method=method, bounds=bounds,
                          constraints=cons if method == 'SLSQP' else (),
                          options={'maxiter': 3000, 'ftol': 1e-10})
        if (best_result is None) or (result.fun < best_result.fun):
            best_result = result
    except Exception as e:
        print(f"{method} failed: {e}")

# Extract best weights
best_w = np.clip(best_result.x, 0, 1)
best_w = best_w / (best_w.sum() + 1e-12)

blend_auc = -best_result.fun


# Print optimized weights
print("Optimized weights:")
for name, w in zip(blend_names, best_w):
    print(f"  {name:12s}: {w:.4f}")
print(f"Optimized OOF AUC: {blend_auc:.6f}")

# Apply weights to test predictions
test_pred_blend = sum(w * r for w, r in zip(best_w, blend_test))

# Select best
candidates = {
    'blend': (blend_auc, test_pred_blend),
    'stack_lr': (stack_lr_auc, test_stack_lr),
}

# Choose best based on OOF AUC
best_name = max(candidates, key=lambda k: candidates[k][0])
final_auc = candidates[best_name][0]
test_pred = candidates[best_name][1]

print(f"\nUsing ensemble source: {best_name} with OOF AUC {final_auc:.6f}")

submission = pd.DataFrame({'SK_ID_CURR': test_ids.astype(int), 'TARGET': test_pred})
submission.to_csv('/kaggle/working/submission_v6.csv', index=False)
print(f"\n=== FINAL OOF AUC (selection metric): {final_auc:.6f} ===")
print(f"Saved submission. Shape: {submission.shape}")
print(submission.head())